# Part D: Food-order stream with RabbitMQ (run in Google Colab)

## 1. Install and start a local RabbitMQ broker

In [1]:
!apt-get install -y -q rabbitmq-server > /dev/null 2>&1
!service rabbitmq-server start
!pip install -q pika

[ OK ]


## 2. Get the scripts from GitHub

In [2]:
!rm -rf freshcart-pipelines
!git clone -q https://github.com/Shristirajpoot/freshcart-pipelines.git
%cd freshcart-pipelines/stream
!ls

/content/freshcart-pipelines/stream
dispatcher.py  driver.py  send_d1_keys.py  stream_colab.ipynb


## 3. Start six drivers (private queue + binding pattern each)

In [3]:
import subprocess, time
DRIVERS = {"Ravi": "bangalore.koramangala.*", "Priya": "bangalore.indiranagar.*", "Arjun": "bangalore.*.ride",
           "Meena": "bangalore.*.food", "Karthik": "chennai.tnagar.food", "Control": "bangalore.#"}
procs = {}
for name, pattern in DRIVERS.items():
    procs[name] = subprocess.Popen(["python", "-u", "driver.py", name, pattern],
                                   stdout=open(f"driver_{name}.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
for name in DRIVERS:
    print(open(f"driver_{name}.log").read().strip())

[Ravi] waiting on pattern 'bangalore.koramangala.*'
[Priya] waiting on pattern 'bangalore.indiranagar.*'
[Arjun] waiting on pattern 'bangalore.*.ride'
[Meena] waiting on pattern 'bangalore.*.food'
[Karthik] waiting on pattern 'chennai.tnagar.food'
[Control] waiting on pattern 'bangalore.#'


## D1: publish the six routing keys from the paper table and see who received each

In [4]:
!python send_d1_keys.py
time.sleep(2)
import re, json
for row in range(1, 7):
    receivers = []
    for name in DRIVERS:
        if re.search(rf"'d1_row': {row},", open(f"driver_{name}.log").read()):
            receivers.append(name)
    print(f"D1 row {row}: ", receivers or "dropped")

published D1 row 1: key=bangalore.koramangala.food
published D1 row 2: key=bangalore.indiranagar.ride
published D1 row 3: key=bangalore.whitefield.ride
published D1 row 4: key=chennai.tnagar.food
published D1 row 5: key=chennai.whitefield.ride
published D1 row 6: key=bangalore.koramangala
D1 row 1:  ['Ravi', 'Meena', 'Control']
D1 row 2:  ['Priya', 'Arjun', 'Control']
D1 row 3:  ['Arjun', 'Control']
D1 row 4:  ['Karthik']
D1 row 5:  dropped
D1 row 6:  ['Control']


## D2 / D3: run the dispatcher for 2 minutes with all drivers running

In [5]:
!python dispatcher.py 120 | tee dispatcher.log

sent #1 key=bangalore.indiranagar.food body={'id': 1, 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'food', 'ts': '2026-11-15T12:22:50.953359+00:00'}
sent #2 key=chennai.tnagar.ride body={'id': 2, 'city': 'chennai', 'zone': 'tnagar', 'type': 'ride', 'ts': '2026-11-15T12:22:52.954009+00:00'}
sent #3 key=bangalore.indiranagar.food body={'id': 3, 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'food', 'ts': '2026-11-15T12:22:54.954764+00:00'}
sent #4 key=chennai.whitefield.food body={'id': 4, 'city': 'chennai', 'zone': 'whitefield', 'type': 'food', 'ts': '2026-11-15T12:22:56.961750+00:00'}
sent #5 key=chennai.whitefield.ride body={'id': 5, 'city': 'chennai', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-15T12:22:58.962280+00:00'}
sent #6 key=bangalore.indiranagar.ride body={'id': 6, 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'ride', 'ts': '2026-11-15T12:23:00.962918+00:00'}
sent #7 key=chennai.tnagar.ride body={'id': 7, 'city': 'chennai', 'zone': 'tnagar', 'type'

In [6]:
for name in DRIVERS:
    log = open(f"driver_{name}.log").read()
    print(f"{name}: {log.count('-> ACK')} ACKed, {log.count('-> NACK')} NACKed")
print()
print("---- Control driver log (receives every Bangalore request) ----")
print(open("driver_Control.log").read())

Ravi: 7 ACKed, 0 NACKed
Priya: 13 ACKed, 3 NACKed
Arjun: 16 ACKed, 4 NACKed
Meena: 12 ACKed, 1 NACKed
Karthik: 7 ACKed, 0 NACKed
Control: 29 ACKed, 5 NACKed

---- Control driver log (receives every Bangalore request) ----
[Control] waiting on pattern 'bangalore.#'
[Control] GOT   key=bangalore.koramangala.food {'id': 101, 'd1_row': 1, 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-15T12:22:37.950482+00:00'} -> ACK
[Control] GOT   key=bangalore.indiranagar.ride {'id': 102, 'd1_row': 2, 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'ride', 'ts': '2026-11-15T12:22:38.250862+00:00'} -> ACK
[Control] GOT   key=bangalore.whitefield.ride {'id': 103, 'd1_row': 3, 'city': 'bangalore', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-15T12:22:38.551487+00:00'} -> ACK
[Control] GOT   key=bangalore.koramangala {'id': 106, 'd1_row': 6, 'city': 'bangalore', 'zone': 'koramangala', 'type': None, 'ts': '2026-11-15T12:22:39.453090+00:00'} -> ACK
[Control] GOT   key=

In [7]:
for p in procs.values():
    p.terminate()